# Figuras de avaliação e análise por ataque (F4, entrega 4.3)

Consome os artefatos gravados pelo contrato C2 em `results/runs/<dataset>/seed<s>/`:

1. **Tabela diagnóstica**: PR-AUC, F1-macro, matriz de confusão e taxa de detecção por categoria (`metrics.py`).
2. **Curvas Precisão-Revocação** por dataset e comparativa (Figura 4 do artigo).
3. **Distribuição do score de anomalia** por ataque com o limiar $\tau$ em escala log (Figuras 5 a 8) e barras de taxa de detecção.
4. **Projeção t-SNE** dos embeddings do E-GraphSAGE extraídos do checkpoint (Figuras 9 a 12).

Datasets sem `scores_test.npz` em `results/runs/` são ignorados automaticamente.

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from graphids.evaluation import (
    binary_confusion_metrics,
    detection_rate_per_attack,
    extract_embeddings,
    f1_macro,
    load_scores,
    plot_anomaly_score_distribution,
    plot_detection_rates,
    plot_pr_curves_comparison,
    plot_precision_recall_curve,
    plot_tsne_projection,
    pr_auc,
)

# Os caminhos do projeto são relativos à raiz do repositório.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

RUNS_DIR = Path("results/runs")
OUT_DIR = Path("results/figuras")
DATA_DIR = "data"
SEED = 1
# Mude para True apenas se quiser incluir o grafo sintético nos testes visuais.
INCLUDE_SYNTHETIC = False

## Execuções disponíveis

In [ ]:
available_runs: dict[str, Path] = {}
for scores_file in sorted(RUNS_DIR.glob(f"*/seed{SEED}/scores_test.npz")):
    run_path = scores_file.parent
    dataset = run_path.parent.name
    if dataset == "synthetic" and not INCLUDE_SYNTHETIC:
        continue
    available_runs[dataset] = run_path
    print(f"ok  {dataset:25s} -> {run_path}")

if not available_runs:
    print("Nenhuma execução real encontrada em results/runs/. "
          "Rode o treino ou defina INCLUDE_SYNTHETIC=True para validar.")

## Métricas globais e taxa de detecção por ataque (Contrato C4)

In [ ]:
summary_rows = []
for dataset, run_path in available_runs.items():
    data = load_scores(run_path / "scores_test.npz")
    metrics = json.loads((run_path / "metrics.json").read_text())
    tau = float(metrics["threshold"])
    preds = (data["errors"] > tau).astype(int)

    cm = binary_confusion_metrics(data["labels"], preds)
    summary_rows.append(
        {
            "Dataset": dataset,
            "PR-AUC": round(pr_auc(data["labels"], data["errors"]), 4),
            "F1-Macro": round(f1_macro(data["labels"], preds), 4),
            "TPR": round(cm["tpr"], 4),
            "FPR": round(cm["fpr"], 4),
            "Threshold": round(tau, 5),
        }
    )

    rates = detection_rate_per_attack(
        data["errors"], data["attacks"], tau, list(data["attack_names"])
    )
    print(f"\n--- Taxa de detecção por ataque: {dataset} ---")
    rates_df = pd.DataFrame(
        {"Ataque": list(rates.keys()), "Taxa_Deteccao": list(rates.values())}
    ).sort_values("Taxa_Deteccao", ascending=False)
    display(rates_df)

if summary_rows:
    display(pd.DataFrame(summary_rows))

## Figura 4: Curvas Precisão-Revocação

In [ ]:
if available_runs:
    runs_npz = {ds: p / "scores_test.npz" for ds, p in available_runs.items()}
    save_cmp = OUT_DIR / "fig4_curvas_pr_comparacao.pdf"
    fig = plot_pr_curves_comparison(runs_npz, output_path=save_cmp)
    plt.show()
    plt.close(fig)

    for dataset, run_path in available_runs.items():
        out = OUT_DIR / f"fig4_pr_{dataset}.png"
        fig = plot_precision_recall_curve(
            run_path / "scores_test.npz", output_path=out, dataset_name=dataset
        )
        plt.show()
        plt.close(fig)

## Figuras 5 a 8: Distribuição do Score de Anomalia e Detecção por Ataque

In [ ]:
for dataset, run_path in available_runs.items():
    scores_file = run_path / "scores_test.npz"
    metrics_file = run_path / "metrics.json"

    fig_dist = plot_anomaly_score_distribution(
        scores_file,
        metrics_path=metrics_file,
        output_path=OUT_DIR / f"fig_scores_{dataset}.pdf",
        dataset_name=dataset,
    )
    plt.show()
    plt.close(fig_dist)

    fig_dr = plot_detection_rates(
        scores_file,
        metrics_path=metrics_file,
        output_path=OUT_DIR / f"fig_deteccao_{dataset}.png",
        dataset_name=dataset,
    )
    plt.show()
    plt.close(fig_dr)

## Figuras 9 a 12: Projeção t-SNE dos Embeddings do E-GraphSAGE

Requer o checkpoint `model.ckpt` e o dataset/cache correspondente em `DATA_DIR`.

In [ ]:
for dataset, run_path in available_runs.items():
    ckpt_path = run_path / "model.ckpt"
    cfg_path = run_path / "config.yaml"
    if not ckpt_path.is_file():
        print(f"Pulando t-SNE de {dataset}: model.ckpt ausente.")
        continue

    data_dir_arg = None if dataset == "synthetic" else DATA_DIR
    emb, atk, names = extract_embeddings(
        config_path=cfg_path,
        checkpoint_path=ckpt_path,
        split="test",
        max_samples=4000,
        seed=SEED,
        data_dir=data_dir_arg,
    )
    fig_tsne = plot_tsne_projection(
        emb,
        atk,
        names,
        output_path=OUT_DIR / f"fig_tsne_{dataset}.pdf",
        max_samples=4000,
        dataset_name=dataset,
    )
    plt.show()
    plt.close(fig_tsne)